In [2]:

import numpy as np

# Data teks dari soal
text1 = [
    "apple", "banana", "orange", "banana",
    "apple", "orange", "banana"
]

text2 = [
    "apple", "banana", "orange", "banana",
    "apple", "orange", "banana", "melon",
    "apple", "banana", "orange", "melon"
]

# Membuat vocabulary
vocab = sorted(set(text1 + text2))
word_to_idx = {word: i for i, word in enumerate(vocab)}
idx_to_word = {i: word for word, i in word_to_idx.items()}

# Parameter RNN
np.random.seed(42)
input_size = len(vocab)
hidden_size = 4
output_size = len(vocab)
learning_rate = 0.01

Wxh = np.random.randn(hidden_size, input_size) * 0.1
Whh = np.random.randn(hidden_size, hidden_size) * 0.1
Why = np.random.randn(output_size, hidden_size) * 0.1
bh = np.zeros((hidden_size, 1))
by = np.zeros((output_size, 1))

def softmax(x):
    x = x - np.max(x)
    exp_x = np.exp(x)
    return exp_x / np.sum(exp_x)

def one_hot(index):
    result = np.zeros((input_size, 1))
    result[index] = 1
    return result

def run_rnn(words):
    h = np.zeros((hidden_size, 1))
    total_loss = 0
    results = []

    for t in range(len(words) - 1):
        current_word = words[t]
        target_word = words[t + 1]

        x = one_hot(word_to_idx[current_word])

        # Hidden state
        h = np.tanh(
            np.dot(Wxh, x) +
            np.dot(Whh, h) + bh
        )

        # Output
        output = np.tanh(np.dot(Why, h) + by)

        # Probabilitas prediksi
        probabilities = softmax(output)

        predicted_idx = np.argmax(probabilities)
        predicted_word = idx_to_word[predicted_idx]

        # Loss berdasarkan target sebenarnya
        target_idx = word_to_idx[target_word]
        loss = -np.log(probabilities[target_idx] + 1e-12)
        total_loss += loss

        results.append({
            "t": t + 1,
            "input": current_word,
            "target": target_word,
            "hidden_state": h.flatten().copy(),
            "output_tanh": output.flatten().copy(),
            "predicted": predicted_word,
            "loss": loss
        })

    return results, total_loss / (len(words) - 1)


def print_results(words, title):
    results, avg_loss = run_rnn(words)

    print("=" * 80)
    print(title)
    print("=" * 80)

    for r in results:
        print(f"\nLangkah ke-{r['t']}")
        print(f"Input Word     : {r['input']}")
        print(f"Target Word    : {r['target']}")
        print(f"Hidden State   : {np.round(r['hidden_state'], 4)}")
        print(f"Output (tanh)  : {np.round(r['output_tanh'], 4)}")
        print(f"Predicted Word : {r['predicted']}")

        # Mengubah loss menjadi angka tunggal
        loss_value = float(np.asarray(r['loss']).item())
        print(f"Loss           : {loss_value:.4f}")

    avg_loss_value = float(np.asarray(avg_loss).item())
    print(f"\nAverage Loss   : {avg_loss_value:.4f}\n")


# Jalankan kedua kasus
print_results(text1, "HASIL KASUS 1")
print_results(text2, "HASIL KASUS 2")



HASIL KASUS 1

Langkah ke-1
Input Word     : apple
Target Word    : banana
Hidden State   : [ 0.0496 -0.0234 -0.0469  0.0242]
Output (tanh)  : [-0.0044  0.0123  0.0031 -0.0009]
Predicted Word : banana
Loss           : 1.3765

Langkah ke-2
Input Word     : banana
Target Word    : orange
Hidden State   : [-0.0187 -0.0194  0.0575 -0.1842]
Output (tanh)  : [ 0.0293 -0.0079  0.0032 -0.018 ]
Predicted Word : apple
Loss           : 1.4061

Langkah ke-3
Input Word     : orange
Target Word    : banana
Hidden State   : [ 0.1726  0.1007 -0.0592 -0.0919]
Output (tanh)  : [-0.0045 -0.0101  0.0179 -0.0397]
Predicted Word : melon
Loss           : 1.3875

Langkah ke-4
Input Word     : banana
Target Word    : apple
Hidden State   : [-0.0098  0.0123  0.0493 -0.2147]
Output (tanh)  : [ 0.029  -0.0134  0.0054 -0.0244]
Predicted Word : apple
Loss           : 1.3567

Langkah ke-5
Input Word     : apple
Target Word    : orange
Hidden State   : [ 0.0767  0.0058 -0.0599 -0.0183]
Output (tanh)  : [-0.0034  0.00